# 💧 02 — Hydrogeological Potential Zone Mapping (AHP-WIO)
> **Pipeline complet : AHP Saaty, traitement du relief sans effet de bord, et classification Jenks**

Ce notebook implémente l'ensemble du pipeline scientifique pour produire la carte nationale de potentiel.


In [ ]:
from pathlib import Path
import yaml
import sys
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_bounds
from rasterio.features import rasterize

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT))

with open(ROOT / 'config' / 'config_niger.yaml', 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

TARGET_CRS = cfg['study_area']['target_crs']
RES = cfg['study_area']['resolution']
print(f"Paramètres : CRS={TARGET_CRS}, Résolution={RES}m")


## 1. Dérivation des poids par la méthode AHP (Saaty)


In [ ]:
from src.hydromap.ahp import AHPModel

ahp = AHPModel(
    criteria=cfg['ahp']['criteria'],
    pairwise_matrix=cfg['ahp']['pairwise_matrix']
)
print(ahp.summary())
assert ahp.is_consistent, 'Attention : Matrice AHP incohérente !'
weights = ahp.get_weights()


## 2. Définition de la grille et du masque national


In [ ]:
gadm_file = ROOT / cfg['paths']['gadm_path']
country = gpd.read_file(gadm_file, layer=cfg['paths']['gadm_country_layer']).to_crs(TARGET_CRS)
regions = gpd.read_file(gadm_file, layer=cfg['paths']['gadm_region_layer']).to_crs(TARGET_CRS)

bounds = country.total_bounds
WIDTH = int((bounds[2] - bounds[0]) / RES)
HEIGHT = int((bounds[3] - bounds[1]) / RES)
TRANSFORM = from_bounds(bounds[0], bounds[1], bounds[2], bounds[3], WIDTH, HEIGHT)

mask = rasterize(
    [(country.union_all(), 1)],
    out_shape=(HEIGHT, WIDTH),
    transform=TRANSFORM,
    fill=0,
    dtype='uint8'
).astype(bool)
print(f"Grille : {WIDTH}x{HEIGHT} pixels | Superficie d'étude : {(mask.sum() * (RES/1000)**2):,.0f} km²")


## 3. Traitement du relief sans effets de bord (Pente & TPI)


In [ ]:
from src.hydromap.terrain import compute_slope, compute_tpi

# Exemple de MNT projeté ou chargement du fichier DEM
dem_path = ROOT / cfg['paths']['dem_path']
if dem_path.exists():
    with rasterio.open(dem_path) as src:
        dem_data = np.zeros((HEIGHT, WIDTH), dtype='float32')
        reproject(rasterio.band(src, 1), dem_data, src_transform=src.transform,
                  src_crs=src.crs, dst_transform=TRANSFORM, dst_crs=TARGET_CRS,
                  resampling=Resampling.bilinear)
else:
    print('[INFO] Utilisation d un relief simulé pour la démonstration.')
    y_grid, x_grid = np.mgrid[0:HEIGHT, 0:WIDTH]
    dem_data = 250.0 + 300.0 * (y_grid / HEIGHT) + 150.0 * np.sin(x_grid / 20.0)

slope_deg, slope_norm = compute_slope(dem_data, resolution_m=RES, mask=mask, invert=True)
tpi_raw, tpi_norm = compute_tpi(dem_data, window_size=15, mask=mask, invert=True)
print('[OK] Pente et TPI calculés sans artefact de bordure.')


## 4. Combinaison multicritère GWPI & Classification Fisher-Jenks


In [ ]:
from src.hydromap.overlay import compute_gwpi, classify_potential, calculate_class_areas

# Création du dictionnaire des couches normalisées
layers = {
    'geology': np.where(mask, 0.75, np.nan),    # Exemple / données BGS
    'rainfall': np.where(mask, 0.50, np.nan),   # Exemple / CHIRPS
    'slope': slope_norm,
    'tpi': tpi_norm
}

gwpi = compute_gwpi(layers, weights, mask=mask)
classes, thresholds = classify_potential(gwpi, method=cfg['classification']['method'], n_classes=4, mask=mask)

print(f'Seuils de classification : {[round(t, 4) for t in thresholds]}')
areas = calculate_class_areas(classes, resolution_m=RES, mask=mask)
for c, stats in areas.items():
    print(f"  Classe {c} ({stats['label']}) : {stats['area_km2']:>10,.0f} km² ({stats['percentage']:.1f}%)")


## 5. Export de la carte haute définition


In [ ]:
from src.hydromap.visualizer import plot_groundwater_potential_map

out_map = ROOT / cfg['paths']['output_maps_dir'] / 'groundwater_potential_map.png'
fig = plot_groundwater_potential_map(
    classes_grid=classes,
    bounds=bounds,
    country_gdf=country,
    regions_gdf=regions,
    target_crs=TARGET_CRS,
    weights=weights,
    cr_score=ahp.cr,
    resolution_m=RES,
    output_path=out_map,
    title='Groundwater Potential Zone Map — Niger',
    subtitle='AHP-WIO Multi-Criteria Model'
)
print(f'[OK] Carte enregistrée sous : {out_map}')
